### Import Libraries

In [1]:
import os
import re
import sys
import json
import time
import csv
import yaml
import warnings
warnings.filterwarnings('ignore')

In [2]:
from typing import List, Optional
from pydantic import BaseModel, Field
from crewai import Agent, Task, Crew, Process
from crewai import LLM
from crewai_tools import (ScrapeWebsiteTool, WebsiteSearchTool, SerperDevTool, 
                          FileReadTool, FileWriterTool, CSVSearchTool, CodeInterpreterTool, NL2SQLTool)
from crewai.tools import tool, BaseTool
from crewai.knowledge.source.pdf_knowledge_source import PDFKnowledgeSource
from crewai.knowledge.source.csv_knowledge_source import CSVKnowledgeSource
from crewai.knowledge.source.crew_docling_source import CrewDoclingSource

### Loading Tasks and Agents YAML Files

In [3]:
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

agents_config = configs['agents']
tasks_config = configs['tasks']

### Initialize LLMs

In [14]:
WATSONX_URL = os.environ["WATSONX_URL"] = os.getenv("WATSONX_URL")  
WATSONX_APIKEY = os.environ["WATSONX_APIKEY"] = os.getenv("WATSONX_APIKEY") 
WATSONX_PROJECT_ID = os.environ["WATSONX_PROJECT_ID"] = os.getenv("WATSONX_PROJECT_ID")
WATSONX_MODEL_ID = os.environ["WATSONX_MODEL_ID"] = "watsonx/ibm/granite-3-2-8b-instruct-preview-rc"    # ibm/granite-3-8b-instruct , ibm/granite-3-2-8b-instruct-preview-rc, mistralai/mistral-large
HF_TOKEN = os.environ["HUGGINGFACE_ACCESS_TOKEN"] = os.getenv("HF_TOKEN")

wx_llm = LLM(
    model = WATSONX_MODEL_ID,
    base_url = WATSONX_URL,
    project_id = WATSONX_PROJECT_ID,
	api_key = WATSONX_APIKEY,
    max_tokens = 10000,
    temperature = 0.1
)

### Define Agents

In [ ]:
from pathlib import Path

jobs_sample_file = "jobs.csv"
root_folder = Path("..").resolve()  
found_files = list(root_folder.rglob(jobs_sample_file))
sample_jobs = str(found_files[0])

resume_sample = "sample.pdf"
root_folder = Path("..").resolve()  
found_files = list(root_folder.rglob(resume_sample))
cv_file = str(found_files[0])

In [ ]:
csv_tool = CSVSearchTool(
    csv = sample_jobs,
    config=dict(
        llm=dict(
            provider="huggingface",  # ('openai', 'azure_openai', 'anthropic', 'huggingface', 'cohere', 'together', 'gpt4all', 'ollama', 'jina', 'llama2', 'vertexai', 'google', 'aws_bedrock', 'mistralai', 'clarifai', 'vllm', 'groq', 'nvidia') 
            config=dict(
                model="ibm-granite/granite-3.1-8b-instruct",
            ),
        ),
        embedder=dict(
            provider="huggingface", 
            config=dict(
                model="ibm-granite/granite-embedding-30m-english"
            ),
        ),
    )
)

In [7]:
pdf_search_tool = PDFKnowledgeSource(file_paths=[resume_sample])

In [8]:
employee_profile_analyzer = Agent(
    config = agents_config['employee_profile_analyzer'],
    verbose = True,
    llm = wx_llm,
    allow_delegation = False,
    knowledge_sources = [pdf_search_tool]
)

matcher = Agent(
    config = agents_config['matcher'],
    tools = [csv_tool, FileWriterTool(name = "candidate_info.txt")],
    verbose = True,
    allow_delegation = True
)

agents_list = [employee_profile_analyzer, matcher]

### Define Tasks

In [9]:
employee_analysis_task = Task(
    config = tasks_config['employee_analysis_task'],
    agent = employee_profile_analyzer
)

match_cv_task = Task(
    config = tasks_config['match_cv_task'],
    agent = matcher,
    output_file = "candidate_info.txt"
)

tasks_list = [employee_analysis_task, match_cv_task]

### Run Crew


In [10]:
match_to_proposal_crew = Crew(
    agents = agents_list, 
    tasks = tasks_list, 
    process = Process.sequential,
    verbose = True,
    # process=Process.hierarchical, 
)

In [ ]:
current_details =     {
    "candidate_id": "C001",
    "name": "Yoko",
    "current_role": "Software Engineer",
    "band": "Band 7",
    "team": "IBM Cloud & AI",
    "skills": ["Python", "Java", "Microservices", "RESTful APIs"],
    "certifications": ["AWS Certified Developer", "IBM Cloud Professional"],
    "training_history": ["Advanced Java Training", "Microservices Architecture"],
    "performance_score": 4.5,
    "manager_feedback": "Excellent problem-solving skills and strong technical expertise",
    "location": "San Francisco"
}

inputs = {
    "current_details": current_details,
    "candidate_cv": cv_file,
    "available_jobs_csv": sample_jobs
}

In [ ]:
result = match_to_proposal_crew.kickoff(inputs)

In [ ]:
print(result.raw)